In [2]:
# To use: save as analyze_projections.ipynb (or paste cells into Jupyter)
# Run from: Evaluation/ directory

# ── Cell 1: Imports and load ──────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

df = pd.read_csv("Results/AllAdvice_Projected.csv")
print(df.shape)
print(df.columns.tolist())
print(df["condition"].value_counts())
print(df["model"].value_counts())
print(df["myth_type"].value_counts(dropna=False))

# ── Cell 2: Basic distribution ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
for condition, grp in df.groupby("condition"):
    ax.hist(grp["projection_score"], bins=50, alpha=0.6, label=condition)
ax.axvline(0, color="black", linestyle="--", linewidth=1)
ax.set_xlabel("Projection score (positive = myth-aligned)")
ax.set_ylabel("Count")
ax.set_title("Distribution of projection scores by condition")
ax.legend()
plt.tight_layout()
# plt.savefig("Results/fig_distribution.png", dpi=150)
plt.show()

# ── Cell 3: Shift by model ────────────────────────────────────────────────────
rows = []
for model, grp in df.groupby("model"):
    for condition, cgrp in grp.groupby("condition"):
        rows.append({
            "model": model,
            "condition": condition,
            "mean": cgrp["projection_score"].mean(),
            "se": cgrp["projection_score"].sem(),
            "n": len(cgrp),
        })
summary = pd.DataFrame(rows)
print(summary.pivot(index="model", columns="condition", values="mean").round(4))

# ── Cell 4: Shift = present - absent per model ────────────────────────────────
pivot = summary.pivot(index="model", columns="condition", values="mean")
pivot["shift"] = pivot["myth"] - pivot["original"]
pivot = pivot.sort_values("shift", ascending=False)
print("\nShift (positive = myth-present advice is more myth-aligned):")
print(pivot[["original", "myth", "shift"]].round(4))

fig, ax = plt.subplots(figsize=(7, 4))
colors = ["#d73027" if s > 0 else "#4575b4" for s in pivot["shift"]]
ax.barh(pivot.index, pivot["shift"], color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Projection shift (myth_present − myth_absent)")
ax.set_title("Myth-alignment shift by model")
plt.tight_layout()
# plt.savefig("Results/fig_shift_by_model.png", dpi=150)
plt.show()

# Cell 5: Shift by myth type
original_mean = df[df["condition"] == "original"]["projection_score"].mean()
rows = []
for myth_type, grp in df[df["condition"] == "myth"].dropna(subset=["myth_type"]).groupby("myth_type"):
    myth_mean = grp["projection_score"].mean()
    rows.append({
        "myth_type": myth_type,
        "myth": myth_mean,
        "original": original_mean,
        "shift": myth_mean - original_mean,
        "n": len(grp),
    })
mt_df = pd.DataFrame(rows).set_index("myth_type")
print(mt_df[["original", "myth", "shift"]].round(4))
fig, ax = plt.subplots(figsize=(7, 4))
colors = ["#d73027" if s > 0 else "#4575b4" for s in mt_df["shift"]]
ax.barh(mt_df.index, mt_df["shift"], color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Projection shift (myth − original baseline)")
ax.set_title("Myth-alignment shift by myth type")
plt.tight_layout()
plt.show()

# Cell 6: Heatmap — model × myth type shift
original_means = df[df["condition"] == "original"].groupby("model")["projection_score"].mean()
rows = []
for (model, myth_type), grp in df[df["condition"] == "myth"].dropna(subset=["myth_type"]).groupby(["model", "myth_type"]):
    shift = grp["projection_score"].mean() - original_means[model]
    rows.append({"model": model, "myth_type": myth_type, "shift": shift})
heatmap_df = pd.DataFrame(rows).pivot(index="model", columns="myth_type", values="shift")
print(heatmap_df.round(4))
fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap(heatmap_df, annot=True, fmt=".3f", center=0,
            cmap="RdBu_r", linewidths=0.5, ax=ax)
ax.set_title("Projection shift: model × myth type")
plt.tight_layout()
plt.show()

# Cell 7: Validity check — perpetrator intoxication
original_means = df[df["condition"] == "original"].groupby("model")["projection_score"].mean()
perp = df[(df["condition"] == "myth") & (df["myth_type"] == "perpetrator_intoxication")]

print("Perpetrator intoxication validity check (shift should be ≤ 0)")
for model, grp in perp.groupby("model"):
    shift = grp["projection_score"].mean() - original_means[model]
    flag = "✓ VALID" if shift <= 0.01 else "✗ FLAG"
    print(f"  {flag}  {model:<15}  shift={shift:+.4f}")
    
# Cell 8: t-test per model
print("t-test: myth vs original projection scores\n")
for model, grp in df.groupby("model"):
    myth_scores = grp[grp["condition"] == "myth"]["projection_score"].values
    orig_scores = grp[grp["condition"] == "original"]["projection_score"].values
    n = min(len(myth_scores), len(orig_scores))
    t, p = stats.ttest_ind(myth_scores, orig_scores)
    shift = myth_scores.mean() - orig_scores.mean()
    d = shift / np.std(np.concatenate([myth_scores, orig_scores]))
    sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
    print(f"  {model:<15}  shift={shift:+.4f}  t={t:+.3f}  p={p:.4f} {sig}  d={d:+.3f}")

# ── Cell 9: Frame comparison ──────────────────────────────────────────────────
# PosMyth > NegMyth > PosNonMyth > NegNonMyth expected
print("Cell 9")
if "frame" in df.columns:
    frame_means = df.groupby(["model", "frame"])["projection_score"].mean().unstack()
    print(frame_means.round(4))

    fig, ax = plt.subplots(figsize=(9, 5))
    frame_means.plot(kind="bar", ax=ax)
    ax.set_xlabel("Model")
    ax.set_ylabel("Mean projection score")
    ax.set_title("Projection score by frame")
    ax.legend(title="Frame", bbox_to_anchor=(1.05, 1))
    plt.tight_layout()
    # plt.savefig("Results/fig_frame_comparison.png", dpi=150)
    plt.show()

# ── Cell 10: Dose comparison ──────────────────────────────────────────────────
if "dose" in df.columns:
    dose_means = df.groupby(["model", "dose", "condition"])["projection_score"].mean()
    print(dose_means.unstack(["dose", "condition"]).round(4))

KeyboardInterrupt: 

In [ ]:
# Cell 8: t-test per model
print("t-test: myth vs original projection scores\n")
for model, grp in df.groupby("model"):
    myth_scores = grp[grp["condition"] == "myth"]["projection_score"].values
    orig_scores = grp[grp["condition"] == "original"]["projection_score"].values
    n = min(len(myth_scores), len(orig_scores))
    t, p = stats.ttest_ind(myth_scores, orig_scores)
    shift = myth_scores.mean() - orig_scores.mean()
    d = shift / np.std(np.concatenate([myth_scores, orig_scores]))
    sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
    print(f"  {model:<15}  shift={shift:+.4f}  t={t:+.3f}  p={p:.4f} {sig}  d={d:+.3f}")

In [3]:
from scipy import stats
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
import pandas as pd
import numpy as np

df = pd.read_csv("Results/AllAdvice_Projected.csv")

# ── Build regression dataset: Cohen's d per model × myth_type × frame × dose
original_means = df[df["condition"] == "original"].groupby("model")["projection_score"].mean()

rows = []
for (model, myth_type, frame, dose), grp in df[df["condition"] == "myth"].dropna(
    subset=["myth_type", "frame", "dose"]
).groupby(["model", "myth_type", "frame", "dose"]):
    myth_scores = grp["projection_score"].values
    orig_scores = df[
        (df["condition"] == "original") & (df["model"] == model)
    ]["projection_score"].values
    pooled_std = np.sqrt((myth_scores.std(ddof=1)**2 + orig_scores.std(ddof=1)**2) / 2)
    if pooled_std == 0:
        continue
    d = (myth_scores.mean() - orig_scores.mean()) / pooled_std
    rows.append({
        "model": model, "myth_type": myth_type,
        "frame": frame, "dose": dose, "cohens_d": d,
    })

reg_df = pd.DataFrame(rows)
print(f"Regression dataset: {len(reg_df)} rows")

# ── Model 5: additive
m5 = smf.ols("cohens_d ~ C(model) + C(myth_type) + C(frame) + dose", data=reg_df).fit()
print(f"\nR² = {m5.rsquared:.4f}  adj-R² = {m5.rsquared_adj:.4f}")
print(m5.summary2().tables[1].round(4))

# ── ANOVA Type II F-tests
anova_table = anova_lm(m5, typ=2)
print("\nANOVA Type II:")
print(anova_table.round(4))

Regression dataset: 160 rows

R² = 0.5751  adj-R² = 0.5435
                                           Coef.  Std.Err.        t   P>|t|  \
Intercept                                -0.1168    0.0204  -5.7323  0.0000   
C(model)[T.llama]                        -0.1207    0.0144  -8.3801  0.0000   
C(model)[T.mistral]                      -0.1096    0.0144  -7.6088  0.0000   
C(model)[T.phi]                          -0.0674    0.0144  -4.6828  0.0000   
C(model)[T.qwen]                         -0.0743    0.0144  -5.1566  0.0000   
C(myth_type)[T.perpetrator_intoxication]  0.0401    0.0129   3.1149  0.0022   
C(myth_type)[T.resistance]                0.1292    0.0129  10.0264  0.0000   
C(myth_type)[T.victim_intoxication]       0.0697    0.0129   5.4135  0.0000   
C(frame)[T.NegNonMyth]                    0.0219    0.0129   1.7036  0.0906   
C(frame)[T.PosMyth]                       0.0094    0.0129   0.7328  0.4648   
C(frame)[T.PosNonMyth]                    0.0213    0.0129   1.6567  0.0

In [4]:
ttest_df["model_dose"] = ttest_df["model"] + "·d" + ttest_df["dose"].astype(int).astype(str)
ttest_df["condition_label"] = ttest_df["myth_type"].str[:5] + "·" + ttest_df["frame"]

pivot     = ttest_df.pivot(index="model_dose", columns="condition_label", values="t_stat")
sig_pivot = ttest_df.pivot(index="model_dose", columns="condition_label", values="significant")

fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(pivot, cmap="RdBu", center=0, ax=ax,
            linewidths=0.3, linecolor="gray",
            cbar_kws={"label": "t-statistic"})

# Significance stars
for i, row in enumerate(pivot.index):
    for j, col in enumerate(pivot.columns):
        if sig_pivot.loc[row, col]:
            ax.text(j + 0.5, i + 0.5, "✶", ha="center", va="center",
                    fontsize=8, color="black", fontweight="bold")

# Myth type dividers
for x in [4, 8, 12]:
    ax.axvline(x, color="black", linewidth=1, linestyle="--")

# Model dividers
for y in [2, 4, 6, 8]:
    ax.axhline(y, color="black", linewidth=1, linestyle="--")

# Myth type labels below x-axis
for label, col in [("Clothing", 2), ("Perp. Intox", 6),
                   ("Resistance", 10), ("Vic. Intox", 14)]:
    ax.text(col, -0.5, label, ha="center", fontsize=9,
            fontweight="bold", va="top")
    
# ax.set_title("Independent t-test: myth vs original advice (BH-corrected, ✶ = significant)")
ax.set_xlabel("")
ax.set_ylabel("")
ax.set_yticklabels(ax.get_yticklabels(), fontsize=10)
# ax.set_xticklabels(ax.get_xticklabels(), fontsize=10, rotation=45, ha="right")

new_labels = [col.split("·")[1] for col in pivot.columns]
ax.set_xticks(range(len(new_labels)))
# ax.set_xticklabels(new_labels, fontsize=10, rotation=45, ha="center")

# ax.set_xticks([x + 0.5 for x in range(len(new_labels))])
ax.set_xticklabels(new_labels, fontsize=10, rotation=45, ha="center")

plt.tight_layout()
# plt.savefig("Results/fig_advice_ttest_heatmap.png", dpi=200, bbox_inches="tight")
plt.show()

NameError: name 'ttest_df' is not defined

In [ ]:
# Add before plt.tight_layout()

